<a href="https://colab.research.google.com/github/sharancode3/CN-Lab-Programs/blob/main/Part_B_Lab_Programs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
def leaky_bucket(cap, process, packets):
    count = 0

    print(f"Bucket Size: {cap}")
    print(f"Processing Rate: {process}")
    print(f"Incoming Packets: {packets}")
    print()
    print(f"{'Second':<10}{'Received':<18}{'Sent':<15}{'Left':<12}{'Dropped'}")
    print("-" * 65)

    for second, packet in enumerate(packets, start=1):
        count += packet
        drop = 0

        if count > cap:
            drop = count - cap
            count = cap

        sent = min(count, process)
        count -= sent

        print(f"{second:<10}{packet:<18}{sent:<15}{count:<12}{drop}")

    second = len(packets) + 1

    while count > 0:
        sent = min(count, process)
        count -= sent

        print(f"{second:<10}{0:<18}{sent:<15}{count:<12}{0}")
        second += 1

    print("\nSimulation completed.\n")


# TEST CASE 1: Normal traffic with packet drops
print("TEST CASE 1: Normal Traffic and Overflow")
leaky_bucket(10, 3, [5, 8, 4, 6, 2])

# TEST CASE 2: No packet loss
print("TEST CASE 2: No Packet Loss")
leaky_bucket(10, 3, [2, 1, 3, 2, 1])

# TEST CASE 3: Packets exceed bucket capacity
print("TEST CASE 3: Bucket Overflow")
leaky_bucket(5, 2, [8, 1, 7, 0])

# TEST CASE 4: Processing rate is higher than incoming traffic
print("TEST CASE 4: High Processing Rate")
leaky_bucket(10, 5, [1, 2, 1])

# TEST CASE 5: Zero incoming packets
print("TEST CASE 5: Zero Incoming Packets")
leaky_bucket(10, 3, [0, 0, 0])

# TEST CASE 6: Bucket reaches its exact capacity
print("TEST CASE 6: Exact Bucket Capacity")
leaky_bucket(6, 2, [6, 0, 0])

# TEST CASE 7: Large packet burst followed by empty seconds
print("TEST CASE 7: Large Burst and Queue Drain")
leaky_bucket(10, 2, [10, 10, 0, 0])

TEST CASE 1: Normal Traffic and Overflow
Bucket Size: 10
Processing Rate: 3
Incoming Packets: [5, 8, 4, 6, 2]

Second    Received          Sent           Left        Dropped
-----------------------------------------------------------------
1         5                 3              2           0
2         8                 3              7           0
3         4                 3              7           1
4         6                 3              7           3
5         2                 3              6           0
6         0                 3              3           0
7         0                 3              0           0

Simulation completed.

TEST CASE 2: No Packet Loss
Bucket Size: 10
Processing Rate: 3
Incoming Packets: [2, 1, 3, 2, 1]

Second    Received          Sent           Left        Dropped
-----------------------------------------------------------------
1         2                 2              0           0
2         1                 1              0         

In [1]:
%%writefile sample.txt
Hello World!
Welcome to Computer Networks Lab.
This is a File Transfer Program.

Writing sample.txt


In [6]:
%%writefile notes.txt
Computer Networks Laboratory
Experiment: File Transfer
Protocol: TCP and UDP
Test case number 2

Writing notes.txt


In [22]:
import socket
import os
import threading

HOST = "127.0.0.1"
PORT = 65432

# Create a sample file if it does not exist
if not os.path.isfile("ksa.txt"):
    with open("ksa.txt", "w") as f:
        f.write("Welcome to Computer Networks Lab.\n")
        f.write("This is a TCP file transfer program.\n")

def tcp_server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind((HOST, PORT))
        server.listen()

        print(f"Server listening on {HOST}:{PORT}")

        while True:
            conn, addr = server.accept()

            with conn:
                print(f"Connected by {addr}")
                filename = conn.recv(1024).decode().strip()

                print(f"Client requested file: {filename}")

                if os.path.isfile(filename):
                    with open(filename, "r") as f:
                        content = f.read()
                    conn.sendall(content.encode())
                else:
                    conn.sendall(b"File not found.")

threading.Thread(target=tcp_server, daemon=True).start()

Server listening on 127.0.0.1:65432


In [24]:
import socket

HOST = '127.0.0.1'
PORT = 65432

filename = input("Enter file name to request: ")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
    client_socket.connect((HOST, PORT))
    client_socket.sendall(filename.encode())

    data = client_socket.recv(4096).decode()

print("\n--- File Content ---")
print(data)

Enter file name to request: sample.txt
Connected by ('127.0.0.1', 49710)
Client requested file: sample.txt

--- File Content ---
Hello World!
Welcome to Computer Networks Lab.
This is a File Transfer Program.



In [4]:
import socket
import os
import threading
import time

HOST = "127.0.0.1"
PORT = 5005

def udp_server():
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as server_socket:
        server_socket.bind((HOST, PORT))

        print(f"UDP Server listening on {HOST}:{PORT}")

        while True:
            filename, client_addr = server_socket.recvfrom(1024)
            filename = filename.decode()

            print(f"Client requested file: {filename}")

            if os.path.isfile(filename):
                with open(filename, "r") as f:
                    data = f.read()

                server_socket.sendto(data.encode(), client_addr)
            else:
                server_socket.sendto(
                    b"ERROR: File not found", client_addr
                )

threading.Thread(target=udp_server, daemon=True).start()
time.sleep(1)

UDP Server listening on 127.0.0.1:5005


In [26]:
import socket

HOST = "127.0.0.1"
PORT = 5005

client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

filename = input("Enter filename to request: ")

client_socket.sendto(filename.encode(), (HOST, PORT))

data, _ = client_socket.recvfrom(4096)

print("\n--- File Content ---")
print(data.decode())

client_socket.close()

Enter filename to request: sample.txt
Client requested file: sample.txt

--- File Content ---
Hello World!
Welcome to Computer Networks Lab.
This is a File Transfer Program.

